In [2]:
from pathlib import Path
import pandas as pd

# Resolve data from the repository root, including kernels started in notebooks/.
working_dir = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / "forecasting" / "paths.py").is_file()
     and (candidate / "notebooks").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the SPDCallDashboard repository.")

run_dir = (
    project_root / "data" / "backtest" / "xgboost"
    / "finalist_test_benchmark" / "20260901T231233Z"
)

metrics = pd.read_parquet(
    run_dir / "neighborhood_metrics.parquet"
)

metrics.columns

Index(['fold', 'feature_set', 'neighborhood', 'n_model_train', 'n_mase_train',
       'n_validation', 'mae', 'rmse', 'mase', 'smape', 'bias', 'config_id',
       'tuning_rank', 'locked_candidate'],
      dtype='str')

In [3]:
locked = metrics.loc[
    metrics["config_id"].astype(str) == "be7924a5110a"
].copy()

In [5]:
locked.sort_values(
    "smape",
    ascending=False
)[
    [
        "neighborhood",
        "smape",
        "mase",
        "mae",
        "rmse",
        "bias",
    ]
]

,neighborhood,smape,mase,mae,rmse,bias
187,COMMERCIAL HARBOR ISLAND,158.547685,0.631596,0.620728,0.676645,0.426718
217,PIGEON POINT,107.751631,0.769052,0.873851,1.079625,0.113100
189,EASTLAKE - EAST,88.290102,0.862691,0.941657,1.183915,0.069015
186,COMMERCIAL DUWAMISH,80.432080,0.966675,1.705818,2.453709,0.151789
191,FAUNTLEROY SW,74.476494,0.727154,1.395755,1.724431,0.184691
228,SOUTH DELRIDGE,64.247501,0.644755,1.486975,1.904694,0.218531
203,MADISON PARK,64.210338,0.711495,1.259440,1.568677,0.171454
194,GENESEE,57.455044,0.761996,1.612096,2.041261,0.218630
199,HILLMAN CITY,54.564152,0.670455,1.456728,1.818386,0.146460
202,LAKEWOOD/SEWARD PARK,50.839838,0.795455,2.074315,2.695239,0.121888


In [6]:
ranked = (
    locked
    .sort_values("smape", ascending=False)
    .reset_index(drop=True)
)

without_worst_5 = ranked.iloc[5:]

without_worst_5["smape"].mean()

np.float64(30.938772571253136)

In [10]:
results = []

for n_removed in range(0, 57):
    remaining = ranked.iloc[n_removed:]

    results.append(
        {
            "worst_neighborhoods_removed": n_removed,
            "neighborhoods_remaining": len(remaining),
            "mean_smape": remaining["smape"].mean(),
            "mean_mase": remaining["mase"].mean(),
            "mean_mae": remaining["mae"].mean(),
            "mean_rmse": remaining["rmse"].mean(),
            "mean_bias": remaining["bias"].mean(),
        }
    )

sensitivity = pd.DataFrame(results)

sensitivity

,worst_neighborhoods_removed,neighborhoods_remaining,mean_smape,mean_mase,mean_mae,mean_rmse,mean_bias
0,0,58,37.056085,0.777653,3.490099,4.465492,-0.018347
1,1,57,34.924654,0.780216,3.540439,4.531963,-0.026155
2,2,56,33.624172,0.780415,3.588057,4.593612,-0.028641
3,3,55,32.630246,0.778919,3.636173,4.655607,-0.030417
4,4,54,31.745027,0.775442,3.671921,4.696383,-0.033791
5,5,53,30.938773,0.776353,3.714867,4.752457,-0.037913
6,6,52,30.298220,0.778884,3.757711,4.807222,-0.042845
7,7,51,29.633277,0.780205,3.806697,4.870723,-0.047047
8,8,50,29.076841,0.780570,3.850589,4.927312,-0.052361
9,9,49,28.556692,0.782817,3.899443,4.990759,-0.056418


In [8]:
predictions = pd.read_parquet(
    run_dir / "predictions.parquet"
)

locked_predictions = predictions.loc[
    predictions["config_id"].astype(str) == "be7924a5110a"
].copy()

one_neighborhood = locked_predictions.loc[
    locked_predictions["neighborhood"] == "SOME NEIGHBORHOOD"
]

one_neighborhood[
    ["target_date", "actual", "prediction"]
]

,target_date,actual,prediction
